# Blog Post Refiner | Prompt Chaining

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Define shared state
class ChainState(TypedDict):
    input: str
    draft: str
    analysis: str
    final_output: str

In [5]:
# Step 1: Generate a draft
def generate_draft(state: ChainState) -> dict:
    response = model.invoke(
        f"Write a short blog post about: {state['input']}"
    )
    return {"draft": response.content}

In [6]:
# Step 2: Analyze the draft
def analyze_draft(state: ChainState) -> dict:
    response = model.invoke(
        f"Analyze this blog post for clarity and tone:\n\n{state['draft']}"
    )
    return {"analysis": response.content}

In [7]:
# Step 3: Produce final version
def finalize(state: ChainState) -> dict:
    response = model.invoke(
        f"Rewrite this blog post incorporating the feedback.\n\n"
        f"Draft:\n{state['draft']}\n\nFeedback:\n{state['analysis']}"
    )
    return {"final_output": response.content}

In [8]:
# Build the chain
graph = StateGraph(ChainState)
graph.add_node("generate", generate_draft)
graph.add_node("analyze", analyze_draft)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate")
graph.add_edge("generate", "analyze")
graph.add_edge("analyze", "finalize")
graph.add_edge("finalize", END)

chain = graph.compile()

In [9]:
# Plot the workflow
plot_mermaid(chain)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	analyze(analyze)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	analyze --> finalize;
	generate --> analyze;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [10]:
result = chain.invoke({"input": "AI agents in production"})
print(result["final_output"])

**Title: Exploring the Future of AI Agents in Production**

As we delve further into the digital realm, artificial intelligence (AI) agents are surfacing as pivotal forces, revolutionizing industries and transforming our approaches to accomplishing tasks. These sophisticated agents, adept at automating intricate processes, analyzing extensive data sets, and making well-informed decisions, are increasingly being integrated into production environments globally.

### The Rise of AI Agents

The recent surge in AI capabilities has been nothing short of transformative. Propelled by advancements in machine learning, natural language processing, and neural networks, AI agents have transitioned from theoretical concepts to essential components in production settings. These agents are now deployed across various sectors, from manufacturing and finance to healthcare and service industries, driving operations and enhancing efficiency.

### Applications in Production

1. **Manufacturing:** AI agen

In [11]:
# Streaming

stream_invoke(chain, {"input": "AI agents in production"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'input': 'AI agents in production',
 'draft': "**Title: Navigating the Era of AI Agents in Production**\n\nAs we sail deeper into the digital age, artificial intelligence (AI) agents are emerging as transformative forces, reshaping industries and redefining how we accomplish tasks. These powerful agents, capable of automating complex processes, analyzing vast data sets, and making informed decisions, are increasingly finding homes in production environments across the globe.\n\n### The Rise of AI Agents\n\nIn recent years, the surge in AI capabilities has been nothing short of revolutionary. Driven by advances in machine learning, natural language processing, and neural networks, AI agents have evolved from mere theoretical constructs to vital components in production settings. These agents are now deployed in various sectors, from manufacturing and finance to healthcare and customer service, to streamline operations and enhance efficiency.\n\n### Applications in Production\n\n1. **Ma